# FRTB SA GIRR — end-to-end demo

CRR3 Art. 325bd / 325bf / 325e / 325ef — delta, vega, and curvature capital charges under the Standardised Approach for General Interest Rate Risk.
<small>

| Capital charge | Regulatory article | Calculator |
|---|---|---|
| GIRR delta | CRR3 Art. 325bd/bf | `SA_GIRR_Calculator` |
| GIRR vega  | CRR3 Art. 325bd/bf | `SA_GIRR_Vega_Calculator` |
| GIRR curvature | CRR3 Art. 325e/ef | `SA_GIRR_Curvature_Calculator` |

## 1. Imports

In [ ]:
import dataclasses

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from banking_risk.frtb.portfolio import (
    FRTB_Risk_Class,
    Trading_Instrument,
    Standard_Trading_Portfolio,
)
from banking_risk.frtb.vertex_mapping import (
    FRTB_GIRR_VERTICES,
    FRTB_GIRR_LABELS,
    GIRR_VEGA_VERTICES,
    GIRR_VEGA_LABELS,
    assign_to_bucket,
    nearest_vertex,
)
from banking_risk.frtb.girr.delta     import SA_GIRR_Calculator
from banking_risk.frtb.girr.vega      import SA_GIRR_Vega_Calculator
from banking_risk.frtb.girr.curvature import (
    SA_GIRR_Curvature_Calculator,
    curvature_pnl_from_greeks,
)
from banking_risk.frtb.constants      import FRTB_GIRR_RISK_WEIGHTS
from banking_risk.shared.curves           import QL_Curve_Adapter
from banking_risk.shared.curve_projection import Curve_Projection
from banking_risk.utils.reporting         import Dark_Style

## 2. Curve construction

Build a EUR zero curve from a tenor/rate array, project it onto all three grids, and inspect the 10 prescribed FRTB GIRR vertices (CRR3 Art. 325bd Table 3).

In [ ]:
tenors_y   = [0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0, 30.0]
base_rates = [0.032, 0.033, 0.034, 0.035, 0.036, 0.037, 0.038, 0.039, 0.040, 0.040, 0.039]

base_curve = QL_Curve_Adapter.from_arrays(tenors_y, base_rates, "2026-06-01")
proj       = Curve_Projection(base_curve)

pd.DataFrame(dataclasses.asdict(proj.frtb)).set_index("labels").round(6)

In [ ]:
Dark_Style().apply()
p = Dark_Style().palette

fig, ax = plt.subplots()
ax.plot(proj.plot.vertices, proj.plot.rates * 100,
        color=p["cyan"], lw=1.8, label="smooth curve")
ax.scatter(proj.frtb.vertices, proj.frtb.rates * 100,
           color=p["amber"], s=60, zorder=5, label="FRTB vertices")

for v, r, lbl in zip(proj.frtb.vertices, proj.frtb.rates, proj.frtb.labels):
    ax.annotate(lbl, (v, r * 100), textcoords="offset points",
                xytext=(0, 8), fontsize=7, color=p["text_muted"], ha="center")

ax.set_xlabel("Tenor (years)")
ax.set_ylabel("Zero rate (%)")
ax.set_title("EUR zero curve — 10 FRTB GIRR vertices (CRR3 Art. 325bd Table 3)")
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()

## 3. Portfolio

In production, instruments come from quant-risk-engine (OIS-discounted pricers).
Here we define minimal stubs that satisfy the `rate_sensitivities(curve, tenors)` and
`vega_grid(curve, expiry_vertices, tenor_vertices)` interfaces so the portfolio
and calculators can run without pricing.

#### Stub instrument interfaces
| Method | Returns | Used by |
|---|---|---|
| `rate_sensitivities(curve, tenors)` | `dict[float, float]` | delta and CSR calculators |
| `vega_grid(curve, expiry_v, tenor_v)` | `dict[tuple[float,float], float]` | vega calculators |

In [ ]:
class _Rate_Stub:
    """Sensitivity concentrated at a single tenor, mapped to the nearest GIRR vertex."""
    def __init__(self, maturity_y: float, sensitivity: float):
        self._mat = maturity_y
        self._s   = sensitivity

    def rate_sensitivities(self, curve, tenors: list[float]) -> dict[float, float]:
        nearest = nearest_vertex(self._mat, tenors)
        return {nearest: self._s}


class _Vega_Stub:
    """Uniform vega across every node of the prescribed expiry × tenor grid."""
    def __init__(self, vega_per_node: float = 500.0):
        self._v = vega_per_node

    def rate_sensitivities(self, curve, tenors):
        return {}

    def vega_grid(self, curve, expiry_vertices, tenor_vertices):
        return {(e, t): self._v for e in expiry_vertices for t in tenor_vertices}

In [ ]:
# Two EUR GIRR instruments (IRS payer + 5Y bond) and one USD IRS.
irs_eur = Trading_Instrument(
    name="IRS_EUR_10Y",
    currency="EUR",
    risk_classes=frozenset({FRTB_Risk_Class.GIRR}),
    instrument=_Rate_Stub(maturity_y=10.0, sensitivity=-1_500.0),
)
bond_eur = Trading_Instrument(
    name="Bond_EUR_5Y",
    currency="EUR",
    risk_classes=frozenset({FRTB_Risk_Class.GIRR}),
    instrument=_Rate_Stub(maturity_y=5.0, sensitivity=800.0),
)
irs_usd = Trading_Instrument(
    name="IRS_USD_3Y",
    currency="USD",
    risk_classes=frozenset({FRTB_Risk_Class.GIRR}),
    instrument=_Rate_Stub(maturity_y=3.0, sensitivity=-600.0),
)
swaption_eur = Trading_Instrument(
    name="Swaption_EUR_5Yx5Y",
    currency="EUR",
    risk_classes=frozenset({FRTB_Risk_Class.GIRR}),
    instrument=_Vega_Stub(vega_per_node=500.0),
)

portfolio = Standard_Trading_Portfolio([irs_eur, bond_eur, irs_usd])
print("Instruments:", [ti.name for ti in portfolio.instruments()])

## 4. GIRR delta

`portfolio.girr_delta_sensitivities(curve)` filters GIRR instruments, calls
`rate_sensitivities(curve, FRTB_GIRR_VERTICES)` on each, maps each sensitivity to its
nearest prescribed vertex via `assign_to_bucket`, and aggregates by currency.
The resulting `dict[str, np.ndarray]` shape is `(10,)` per currency — exactly what
`SA_GIRR_Calculator.compute()` expects.

In [ ]:
delta_sens = portfolio.girr_delta_sensitivities(base_curve)

print("Raw sensitivities on the 10 GIRR vertices:")
for ccy, arr in delta_sens.items():
    print(f"\n  {ccy}  (shape {arr.shape})")
    for label, val in zip(FRTB_GIRR_LABELS, arr):
        marker = " ◀" if val != 0.0 else ""
        print(f"    {label:>6}  {val:>10.2f}{marker}")

In [ ]:
# Vertex mapping: show the assignment for one off-grid sensitivity
print("nearest_vertex(2.6, FRTB_GIRR_VERTICES) =", nearest_vertex(2.6, FRTB_GIRR_VERTICES), "Y")
print("nearest_vertex(4.9, FRTB_GIRR_VERTICES) =", nearest_vertex(4.9, FRTB_GIRR_VERTICES), "Y")

# assign_to_bucket returns (10,) array with sums at each vertex
raw = {2.6: 50.0, 4.9: -30.0}
bucket = assign_to_bucket(raw, FRTB_GIRR_VERTICES)
pd.Series(bucket, index=FRTB_GIRR_LABELS, name="WS example")[bucket != 0]

In [ ]:
delta_result = SA_GIRR_Calculator().compute(delta_sens)

print(f"Currencies   : {delta_result.currencies}")
print(f"K (EUR)      : {delta_result.K['EUR']:.4f}")
print(f"K (USD)      : {delta_result.K['USD']:.4f}")
print(f"S (EUR)      : {delta_result.S['EUR']:.4f}")
print(f"S (USD)      : {delta_result.S['USD']:.4f}")
print(f"Delta capital: {delta_result.capital:.4f}")

In [ ]:
# WS table: rows = vertices, cols = currencies; K and S appended at bottom
delta_result.to_table().round(4)

In [ ]:
delta_result.plot()

## 5. GIRR vega

Options (swaptions, caps/floors, callable bonds) carry vega risk — sensitivity to
changes in implied volatility. CRR3 Art. 325bd prescribes a **5 × 5** grid of
(option expiry × underlying tenor) nodes.

The flat risk weight is **0.4%** across all GIRR nodes (Art. 325bd Table 3).
Within-bucket correlation uses a relative-difference decay formula (Art. 325bf);
the full (25 × 25) correlation matrix is the Kronecker product
RHO = RHO_expiry ⊗ RHO_tenor.

In [ ]:
vega_portfolio = Standard_Trading_Portfolio([swaption_eur])

# girr_vega_sensitivities returns flat (25,) per currency.
# SA_GIRR_Vega_Calculator expects shape (5, 5) — reshape before passing.
N_VEGA     = len(GIRR_VEGA_VERTICES)
vega_flat  = vega_portfolio.girr_vega_sensitivities(base_curve)
vega_grids = {ccy: arr.reshape(N_VEGA, N_VEGA) for ccy, arr in vega_flat.items()}

print("Vega grid shape (EUR):", vega_grids["EUR"].shape, "  (expiry × tenor)")
pd.DataFrame(
    vega_grids["EUR"],
    index=pd.Index(GIRR_VEGA_LABELS, name="expiry"),
    columns=pd.Index(GIRR_VEGA_LABELS, name="tenor"),
).round(2)

In [ ]:
vega_result = SA_GIRR_Vega_Calculator().compute(vega_grids)

print(f"Currencies   : {vega_result.currencies}")
print(f"K (EUR)      : {vega_result.K['EUR']:.4f}")
print(f"Vega capital : {vega_result.capital:.4f}")

In [ ]:
# WS heatmap (5×5) — expiry rows, tenor columns
vega_result.to_table().round(6)

In [ ]:
vega_result.plot()

## 6. GIRR curvature

Curvature capital arises from instruments with **non-zero gamma** (second-order rate
sensitivity). Linear instruments (IRS, bonds) have gamma ≈ 0 and contribute no
curvature charge.

The second-order Taylor approximation (CRR3 Art. 325e):

> CVR^± ≈ −0.5 × Σ_k γ_k × ΔRW_k²

where ΔRW_k are the same GIRR delta risk weights (in decimal). Because the formula is
symmetric, CVR^+ = CVR^−.

| Instrument | Gamma sign | CVR sign | Capital |
|---|---|---|---|
| IRS / bond | 0 | 0 | none |
| Long swaption (long gamma) | + | − (gain) | K = 0 |
| Short swaption (short gamma) | − | + (loss) | K = CVR > 0 |

In [ ]:
N  = len(FRTB_GIRR_VERTICES)
rw = np.array(FRTB_GIRR_RISK_WEIGHTS) / 10_000   # bps → decimal

# 1. IRS — linear, zero gamma
delta_irs   = np.zeros(N);  delta_irs[6] = -1_500.0   # 10Y vertex
gamma_irs   = np.zeros(N)
cvr_irs     = curvature_pnl_from_greeks(delta_irs, gamma_irs)

# 2. Long swaption — positive gamma (convexity gain under stress)
delta_long  = np.zeros(N)
gamma_long  = np.zeros(N);  gamma_long[5] = 1e8       # 5Y vertex
cvr_long    = curvature_pnl_from_greeks(delta_long, gamma_long)

# 3. Short swaption — negative gamma (convexity loss → capital charge)
delta_short = np.zeros(N)
gamma_short = np.zeros(N);  gamma_short[5] = -1e8
cvr_short   = curvature_pnl_from_greeks(delta_short, gamma_short)

pd.DataFrame(
    {
        "gamma_5Y" : [0, 1e8, -1e8],
        "CVR_up"   : [cvr_irs[0],   cvr_long[0],   cvr_short[0]],
        "CVR_down" : [cvr_irs[1],   cvr_long[1],   cvr_short[1]],
        "K_charge" : [
            max(cvr_irs[0],   cvr_irs[1],   0),
            max(cvr_long[0],  cvr_long[1],  0),
            max(cvr_short[0], cvr_short[1], 0),
        ],
    },
    index=["IRS (linear)", "Long swaption", "Short swaption"],
).round(6)

In [ ]:
# Portfolio: EUR has IRS + short swaption (net CVR = 0 + CVR_short > 0)
#            USD has the long swaption  (net CVR < 0 → K = 0, no charge)
cvr_up_eur   = cvr_irs[0] + cvr_short[0]
cvr_down_eur = cvr_irs[1] + cvr_short[1]
cvr_up_usd   = cvr_long[0]
cvr_down_usd = cvr_long[1]

curv_result = SA_GIRR_Curvature_Calculator().compute(
    cvr_up  ={"EUR": cvr_up_eur,   "USD": cvr_up_usd},
    cvr_down={"EUR": cvr_down_eur, "USD": cvr_down_usd},
)

print(f"K(EUR) = {curv_result.K['EUR']:.6f}  (short swaption → loss → capital)")
print(f"K(USD) = {curv_result.K['USD']:.6f}   (long swaption  → gain → no charge)")
print(f"Curvature capital: {curv_result.capital:.6f}")

In [ ]:
curv_result.to_table().round(6)

## 7. Combined GIRR capital

CRR3 Art. 325bb: total GIRR capital = GIRR_delta + GIRR_vega + GIRR_curvature.
The three charges are **additive** at the risk-class level (no diversification benefit
across measure types). Diversification applies only within each measure via the
correlation matrices.

In [ ]:
total = delta_result.capital + vega_result.capital + curv_result.capital

summary = pd.DataFrame(
    {
        "Capital": {
            "GIRR delta"    : delta_result.capital,
            "GIRR vega"     : vega_result.capital,
            "GIRR curvature": curv_result.capital,
            "Total GIRR"    : total,
        }
    }
)
summary.index.name = "component"
summary.round(4).style.format("{:.4f}").set_caption("GIRR SA Capital (CRR3 Art. 325bb)")